In [ ]:
NEED TO FIGURE OUT LOADING PROTOCOL

# Conditioning
 - condition sensor 4+ times at a pressure that is 20% greater than expected maximum pressure
    - recommended by tekscan, referenced by Brimacombe (2009)
 - Need to look up details of conditioning

# Equilibration
- Don't know how to apply perfectly even force, so risk introducing more error
    - pneumatic pressure bladder not available
    - Not reasonable to use heavy enough free weights , also this paper mentions that dead weights face similar issues to instron with fixed plates in regards to getting perfectly uniform load ( https://pmc.ncbi.nlm.nih.gov/articles/PMC9571954/ ).
- Need to get calibration within reasonable accuracy to justify not doing this

# Calibration
- Calibrate over expected pressure range
- This paper talks on difficulties of calibration ( https://pmc.ncbi.nlm.nih.gov/articles/PMC9571954/ )
- Ideally do 10 evenly spread calibration frames that cover pressure range
    - 4 repeats at each frame rotating sensor 90 degrees for each repeat 
    - fit accross all frames

---
- Below is some near equivalent fitting methods that reproduce the power law fit coefficients that tekscan computes

# Also:
- Figure out loading protocol that actually remains stable at force levels
- Double check the distance of sensor grid from edge of sensor
- Note any cells that have noisy readings when unloaded

In [25]:
from pathlib import Path
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.optimize import least_squares


from phd_helpers.experiments import parse_tekscan

In [26]:
def signif(x, p):
    """round to specific number of significant figures (from polars github discussion)"""
    x = np.asarray(x)
    x_positive = np.where(np.isfinite(x) & (x != 0), np.abs(x), 10**(p-1))
    mags = 10 ** (p - 1 - np.floor(np.log10(x_positive)))
    return np.round(x * mags) / mags

def N2FP(FN):
    """convert force from Newtions to Force Pounds like tescan does"""
    FP2N = 4.448222 # Force pounds * FP2N = Newtons
    return signif(FN / FP2N, 6)

In [27]:
cal_path = Path(f'/Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/Experiments/TekscanCalibration/20260317/cal-tekscan')
cal = f'cal5'
F1, F2 = 10, 60 # Newtons
A = (0.55**2) / (11**2) # Sensel area (inch^2)
# calibration file
cal_file = cal_path / f'{cal}.cal'
# 10N, 60N raw frames
raw10 = parse_tekscan(cal_path / f'{cal}-{F1}N-raw.asm', sensor=1)
raw60 = parse_tekscan(cal_path / f'{cal}-{F2}N-raw.asm', sensor=1)

FP1, FP2 = N2FP(F1), N2FP(F2) # Force pounds

In [28]:
patches = [raw10, raw60] # calibration patch arrays
Fs = np.array([FP1, FP2], dtype=float) # known total loads
sensel_area = A # area of one sensel

xs = [np.asarray(p, dtype=float)[np.asarray(p) >= 3] for p in patches] # Loaded sensels only

# Average pressure on each patch
n = np.array([len(x) for x in xs], dtype=float)
area = n * sensel_area
p = Fs / area

# Fit: p_i ≈ a * mean(X_i ** b)
def S(b):
    return np.array([np.mean(x ** b) for x in xs])

def a_of_b(b):
    s = S(b)
    w = area ** 2
    return np.dot(w * p, s) / np.dot(w * s, s)

def J(b):
    s = S(b)
    a = a_of_b(b)
    return np.sum((area * (a * s - p)) ** 2)

res = minimize_scalar(J, bounds=(0, 3), method="bounded")

b = res.x
a = a_of_b(b)

print(f"a: {a:.5f}")
print(f"b: {b:.5f}")

a: 2.35885
b: 1.03688


In [29]:
def J_relative(b):
    s = S(b)
    a = a_of_b_relative(b)
    F_pred = a * sensel_area * np.array([np.sum(x ** b) for x in xs])
    return np.sum(((F_pred - Fs) / Fs) ** 2)

def a_of_b_relative(b):
    q = sensel_area * np.array([np.sum(x ** b) for x in xs])
    w = 1 / Fs**2
    return np.dot(w * Fs, q) / np.dot(w * q, q)

res = minimize_scalar(J_relative, bounds=(0, 3), method="bounded")
b = res.x
a = a_of_b_relative(b)

print(f"a: {a:.5f}")
print(f"b: {b:.5f}")

a: 2.35885
b: 1.03688


In [30]:
def residual(theta):
    log_a, b = theta
    a = np.exp(log_a)
    s = np.array([np.mean(x ** b) for x in xs])
    return area * (a * s - p)

res = least_squares(
    residual,
    x0=[np.log(1.0), 1.0],
    bounds=([-np.inf, 0.0], [np.inf, 3.0])
)

a = np.exp(res.x[0])
b = res.x[1]

print(f"a: {a:.5f}")
print(f"b: {b:.5f}")

a: 2.35884
b: 1.03688


In [31]:
res = least_squares(
    residual,
    x0=[np.log(1.0), 1.0],
    bounds=([-np.inf, 0.0], [np.inf, 3.0]),
    loss="soft_l1",
    f_scale=1.0
)

a = np.exp(res.x[0])
b = res.x[1]

print(f"a: {a:.5f}")
print(f"b: {b:.5f}")

a: 2.35884
b: 1.03688
